# Regression Trees and Ensembles — Concepts

Read the idea, the formula and the small example in that order. The demonstration code is provided.

## Libraries and settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.inspection import permutation_importance

## 1. A regression tree predicts a mean

A tree asks yes/no questions such as “engine size ≤ 2 litres?”. Each answer takes us to another node. The final node is a **leaf**. Its prediction is the mean target of training rows in that leaf. A tree divides the input space into regions, with one constant prediction per region.

To choose a split, calculate the squared errors around each child mean:
$$\mathrm{SSE}_{\mathrm{split}}=\sum_{i\in\mathrm{left}}(y_i-\bar y_{\mathrm{left}})^2+\sum_{i\in\mathrm{right}}(y_i-\bar y_{\mathrm{right}})^2.$$
Try candidate features and thresholds; pick the smallest total SSE. Repeat in each child (**recursive**, **greedy** splitting). For targets 10,12,14,30,34, a split after 14 gives predictions 12 and 32 and total SSE 16, compared with 496 before splitting.

In [ ]:
y = np.array([10, 12, 14, 30, 34])
left = y[:3]
right = y[3:]
print('Parent SSE:', np.sum((y - y.mean()) ** 2))
print('Leaf means:', left.mean(), right.mean())
print('Split SSE:', np.sum((left - left.mean()) ** 2) + np.sum((right - right.mean()) ** 2))

## 2. Size, hyperparameters and pruning

Deep trees can learn noise. A **hyperparameter** is a setting we choose before fitting.

| Parameter | Meaning | Effect of increasing it |
|---|---|---|
| `max_depth` | maximum depth | allows a more flexible tree |
| `min_samples_split` | rows needed to split a node | discourages small splits |
| `min_samples_leaf` | minimum rows per leaf | smooths predictions |
| `max_leaf_nodes` | maximum leaves | allows more regions |
| `max_features` | inputs considered at a split | less randomness when more are considered |
| `ccp_alpha` | pruning penalty | smaller tree |
| `criterion` | split measure | regression: squared error; classification: Gini/entropy |

**Pruning** removes branches whose benefit is too small compared with a fee per leaf:
$$\mathrm{pruning\ objective}=\mathrm{tree\ error}+\alpha\times\mathrm{number\ of\ leaves}.$$
Here $\alpha$ is the penalty. `ccp_alpha=0` adds no fee; larger values can remove branches. Scikit-learn expresses the regression error on a per-observation scale, so alpha values depend on the target units. Choose settings with CV on training data.

## 3. Bagging and out-of-bag rows

**Bagging** fits trees on bootstrap samples: draw training rows **with replacement**, so some appear repeatedly and others are omitted. Average the predictions:
$$\hat y_{\mathrm{bag}}=\frac1B\sum_{b=1}^{B}\hat y_b.$$
$B$ is the number of trees. Predictions 8, 10, 9 give an average of 9.

For a large sample, roughly 37% of rows are left out of a given tree's bootstrap sample. These **out-of-bag (OOB)** rows allow an internal evaluation using trees that did not see the row. OOB evaluation does not replace the final test.

Averaging reduces instability. With equal tree variance $\sigma^2$ and common correlation $\rho$:
$$\mathrm{Var}(\mathrm{average})=\rho\sigma^2+\frac{(1-\rho)\sigma^2}{B}.$$
Only the independent part shrinks as we add trees. For $\sigma^2=1$, $\rho=0.2$ and $B=100$, variance is 0.208. No derivation is required.

## 4. Random forest

A random forest adds **random feature subsets at each split** to bagging. This makes trees less similar, so averaging can help more. Use `n_estimators` for the number of trees and `max_features` for the inputs considered at each split.

In scikit-learn, a regression forest defaults to all inputs (`max_features=1.0`); a classification forest defaults to `"sqrt"`. More trees stabilise results, with increasing computation and diminishing gains.

Feature importance asks which inputs help prediction. Impurity importance can favour some inputs unfairly. **Permutation importance** shuffles one feature and measures the decrease in predictive performance. Importance is not a causal effect.

## 5. Gradient boosting and comparison

Boosting builds small trees **one after another**. For squared loss, each new tree learns the current residuals:
$$r_i=y_i-F_{m-1}(x_i),\qquad F_m(x)=F_{m-1}(x)+\nu h_m(x).$$
$F_{m-1}$ is the current model, $h_m$ the next tree and $\nu$ the learning rate. If the current prediction is 7, the next tree predicts residual 3 and $\nu=0.1$, the updated prediction is $7+0.1\times3=7.3$. Other differentiable losses use negative gradients (pseudo-residuals).

| Model | Main idea | Main settings |
|---|---|---|
| Linear regression | weighted sum of inputs | chosen inputs |
| Single tree | readable rules and constant leaf predictions | depth, pruning |
| Random forest | average many varied trees | trees, random features |
| Boosting | correct errors step by step | learning rate, trees, depth |

Bagging mainly reduces variance; boosting reduces systematic error step by step and can affect both bias and variance. Compare validation results: no method always wins.

In [ ]:
old_prediction = 7
residual_prediction = 3
learning_rate = 0.1
print('Updated prediction:', old_prediction + learning_rate * residual_prediction)

### Jupyter notebook — footer info

Include this information when submitting your notebook.

In [ ]:
import platform
from datetime import datetime
print(platform.platform())
print("Python:", platform.python_version())
print("Date:", datetime.now().isoformat(timespec="seconds"))